# Baseline Survival ML Models

Ce notebook a pour objectif de prédire le temps restant avant l'insuffisance rénale terminale (IRT) à partir des caractéristiques cliniques et génétiques disponibles à la baseline, à l'aide de modèles de machine learning adaptés aux données de survie. Les modèles étudiés sont le Random Survival Forest (RSF), le modèle XGBoost Survival AFT.

Le notebook s'appuie sur trois jeux de données déjà constitués : un jeu d'entraînement (`train_df`), utilisé pour la sélection des hyperparamètres par validation croisée et l'entraînement des modèles, un jeu de validation interne (`internal_test_df`), utilisé pour évaluer les performances sur des données indépendantes, et un jeu de validation externe (`external_test_df`), réservé à la validation finale afin d'évaluer la capacité de généralisation des modèles sur une cohorte indépendante.


### Importation des bibliothèques

In [ ]:
# ==========================================================
# IMPORTS
# ==========================================================

# Standard library
import json
import os
from itertools import product

# Data manipulation
import numpy as np
import pandas as pd
import polars as pl

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

# Machine Learning
import optuna
import shap
import xgboost as xgb

# Progress bar
from tqdm.auto import tqdm

# Scikit-learn
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import (
    ParameterGrid,
    StratifiedKFold,
    train_test_split,
)

# Survival analysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import (
    concordance_index_censored,
    concordance_index_ipcw,
)
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.util import Surv

# Statistics
from scipy.stats import kruskal

#### CHARGEMENT DES JEUX DE DONNÉES

In [ ]:
# Chargement des trois jeux de données
# - train_df : jeu de développement utilisé pour l'entraînement
#   et la sélection des modèles.
# - internal_test_df : jeu de validation interne utilisé pour
#   évaluer les performances sur des données indépendantes.
# - external_test_df : jeu de validation externe, réservé à la
#   validation finale des modèles.

train_df = pl.read_parquet("My_data/train.parquet")
internal_test_df = pl.read_parquet("My_data/internal_test.parquet")
external_test_df = pl.read_parquet("My_data/external_test.parquet")


# Conversion des DataFrames Polars en DataFrames pandas
# Certaines bibliothèques de survie (scikit-survival, XGBoost,
# etc.) nécessitent des objets pandas en entrée.

train_df = train_df.to_pandas()
internal_test_df = internal_test_df.to_pandas()
external_test_df = external_test_df.to_pandas()


# Vérification des dimensions des jeux de données

print(f"Train : {train_df.shape[0]} patients, {train_df.shape[1]} variables")
print(f"Test interne : {internal_test_df.shape[0]} patients, {internal_test_df.shape[1]} variables")
print(f"Test externe : {external_test_df.shape[0]} patients, {external_test_df.shape[1]} variables")

### Contrôles de cohérence des données


In [ ]:
# ==========================================================
# VÉRIFICATION DE LA VARIABLE time_to_event
# ==========================================================

print("=" * 40)
print("Résumé de time_to_event")
print("=" * 40)

print("\nTrain")
print(train_df["time_to_event"].describe())

print("\nTest interne")
print(internal_test_df["time_to_event"].describe())

In [ ]:
# ==========================================================
# RECHERCHE DES TEMPS NON POSITIFS
# ==========================================================

train_non_positive = train_df.loc[
    train_df["time_to_event"] <= 0,
    ["patient_id", "time_to_event"]
]

test_non_positive = internal_test_df.loc[
    internal_test_df["time_to_event"] <= 0,
    ["patient_id", "time_to_event"]
]

print(f"\nTrain : {len(train_non_positive)} observations")
display(train_non_positive)

print(f"\nTest : {len(test_non_positive)} observations")
display(test_non_positive)

In [ ]:
# ==========================================================
# RECHERCHE DES VALEURS MANQUANTES
# ==========================================================

print("=" * 40)
print("Valeurs manquantes")
print("=" * 40)

print(f"Train : {train_df['time_to_event'].isna().sum()}")
print(f"Test  : {internal_test_df['time_to_event'].isna().sum()}")

#### Construction des matrices  pour les 3 modèles cliniques

In [ ]:
# ==========================================================
# VARIABLES EXPLICATIVES
# ==========================================================

# Variables communes aux trois modèles
common_features = [
    "sex",
    "creat_age",
    "ckd_epi",
    "genetic"
]

# ----------------------------------------------------------
# Modèle 1 : Variables cliniques au baseline
# ----------------------------------------------------------

features_model1 = common_features + [
    "has_hypertension_baseline",
    "has_urologic_event_baseline",
]

# ----------------------------------------------------------
# Modèle 2 : Événements survenus avant 35 ans
# ----------------------------------------------------------

features_model2 = common_features + [
    "htn_before_35",
    "uro_before_35",
]

# ----------------------------------------------------------
# Modèle 3 : Modèle combiné
# ----------------------------------------------------------

features_model3 = common_features + [
    "has_hypertension_baseline",
    "has_urologic_event_baseline",
    "htn_before_35",
    "uro_before_35",
]

In [ ]:
# ==========================================================
# Fonction de préparation des variables explicatives
# ==========================================================

def prepare_features(train_df, test_df, features):
    """
    Prépare les matrices d'entraînement et de test :
        - sélection des variables
        - encodage des variables catégorielles
        - alignement des colonnes
        - conversion des booléens en entiers
    """

    # Sélection des variables du modèle
    X_train = train_df[features].copy()
    X_test = test_df[features].copy()

    # Encodage one-hot des variables catégorielles
    categorical_features = ["sex", "genetic"]

    X_train = pd.get_dummies(
        X_train,
        columns=categorical_features,
        drop_first=False,
        dtype=np.int8,
    )

    X_test = pd.get_dummies(
        X_test,
        columns=categorical_features,
        drop_first=False,
        dtype=np.int8,
    )


    X_test = X_test.reindex(
        columns=X_train.columns,
        fill_value=0,
    )


    bool_cols = X_train.select_dtypes(include="bool").columns

    if len(bool_cols) > 0:
        X_train[bool_cols] = X_train[bool_cols].astype(np.int8)
        X_test[bool_cols] = X_test[bool_cols].astype(np.int8)

    return X_train, X_test

In [ ]:
# ==========================================================
# Préparation des variables explicatives
# ==========================================================

# Modèle 1 : Baseline
X_train_model1, X_internal_model1 = prepare_features(
    train_df,
    internal_test_df,
    features_model1,
)

# Modèle 2 : Events before 35
X_train_model2, X_internal_model2 = prepare_features(
    train_df,
    internal_test_df,
    features_model2,
)

# Modèle 3 : Combiné
X_train_model3, X_internal_model3 = prepare_features(
    train_df,
    internal_test_df,
    features_model3,
)

In [ ]:
# ==========================================================
# Construction de la cible de survie
# ==========================================================

y_train = Surv.from_dataframe(
    event="event_irt",
    time="time_to_event",
    data=train_df,
)

y_internal_test = Surv.from_dataframe(
    event="event_irt",
    time="time_to_event",
    data=internal_test_df,
)

In [ ]:
# ==========================================================
# Vérification des dimensions
# ==========================================================

print("=" * 30)
print("Dimensions des jeux de données")
print("=" * 30)

for i, (X_train, X_test) in enumerate(
    [
        (X_train_model1, X_internal_model1),
        (X_train_model2, X_internal_model2),
        (X_train_model3, X_internal_model3),
    ],
    start=1,
):
    print(f"\nModèle {i}")
    print(f"Train    : {X_train.shape}")
    print(f"Test     : {X_test.shape}")

In [ ]:
# ==========================================================
# MATRICES DES VARIABLES EXPLICATIVES
# ==========================================================

# Pour chaque modèle, on stocke les matrices
# d'entraînement et de test dans un dictionnaire.

models_dict = {

    "Model 1 - baseline": (
        X_train_model1,
        X_internal_model1,
    ),

    "Model 2 - before35": (
        X_train_model2,
        X_internal_model2,
    ),

    "Model 3 - combined": (
        X_train_model3,
        X_internal_model3,
    ),
}

# ==========================================================
# OBJETS DE SURVIE
# ==========================================================

print("=" * 35)
print("Objets de survie")
print("=" * 35)

print(f"Train : {len(y_train)} observations")
print(f"Test  : {len(y_internal_test)} observations")

#### 1. RANDOM SURVIVAL FOREST

In [ ]:
# ==========================================================
# 2. DÉFINITION DE LA GRILLE D'HYPERPARAMÈTRES DU RSF
# ==========================================================

# Définition des valeurs candidates pour chaque hyperparamètre
# du modèle Random Survival Forest.

param_grid = {
    "n_estimators": [100, 300, 500],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [5, 10, 15],
    "max_features": ["sqrt", "log2"],
}

# Génération de toutes les combinaisons possibles
# d'hyperparamètres à évaluer.

param_grid = list(ParameterGrid(param_grid))

# Nombre total de configurations testées

print(f"Nombre de combinaisons : {len(param_grid)}")

In [ ]:
# ==========================================================
# 3. VALIDATION CROISÉE POUR LA CALIBRATION DES
#    HYPERPARAMÈTRES DU RANDOM SURVIVAL FOREST
# ==========================================================

def cross_validation_rsf(X_train, y_train, params, n_splits=5):
    """
    Évalue une combinaison d'hyperparamètres du Random Survival
    Forest par validation croisée stratifiée à 5 folds.

    À chaque itération, le modèle est entraîné sur quatre folds
    et évalué sur le fold restant. Le critère de sélection est le
    C-index moyen obtenu sur l'ensemble des folds.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Variables explicatives du jeu de développement.

    y_train : structured array (Surv)
        Variable de survie (temps jusqu'à l'événement et statut).

    params : dict
        Combinaison d'hyperparamètres à évaluer.

    n_splits : int, default=5
        Nombre de folds de la validation croisée.

    Returns
    -------
    float
        C-index moyen obtenu sur les cinq folds.
    """

    # Création des folds en conservant la proportion de patients
    # avec et sans événement dans chaque sous-échantillon.

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    # Stockage du C-index obtenu sur chaque fold

    cindex_scores = []

    # Validation croisée sur les différentes partitions

    for train_idx, val_idx in tqdm(
        cv.split(X_train, y_train["event_irt"]),
        total=n_splits,
        desc="5-Fold CV",
        leave=False
    ):

        # Séparation des données d'entraînement et de validation
        # pour le fold courant.

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train[train_idx]
        y_val = y_train[val_idx]

        # Initialisation du modèle avec la combinaison
        # d'hyperparamètres évaluée.

        rsf = RandomSurvivalForest(
            n_estimators=params["n_estimators"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=42,
            n_jobs=-1
        )

        # Entraînement sur les données du fold d'entraînement

        rsf.fit(X_tr, y_tr)

        # Prédiction des scores de risque sur le fold de validation

        risk_scores = rsf.predict(X_val)

        # Calcul du C-index sur le fold de validation

        cindex = concordance_index_censored(
            event_indicator=y_val["event_irt"],
            event_time=y_val["time_to_event"],
            estimate=risk_scores
        )[0]

        # Conservation du score obtenu pour ce fold

        cindex_scores.append(cindex)

    # Retour du C-index moyen sur l'ensemble des folds

    return np.mean(cindex_scores)

In [ ]:
# ==========================================================
# 4. RECHERCHE SUR GRILLE PAR VALIDATION CROISÉE À 5 FOLDS
# ==========================================================

# Dictionnaire contenant, pour chaque modèle, la meilleure
# combinaison d'hyperparamètres et le C-index moyen associé.

best_models = {}

for model_name, (X_train, _) in models_dict.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # Initialisation des meilleures performances obtenues
    # pour le modèle courant.

    best_cindex = -np.inf
    best_params = None

    # Évaluation de chaque combinaison d'hyperparamètres
    # définie dans la grille.

    for i, params in enumerate(param_grid, start=1):

        print(f"\nCombination {i}/{len(param_grid)}")
        print(params)

        # Évaluation de la combinaison par validation croisée
        # stratifiée à 5 folds.

        cindex = cross_validation_rsf(
            X_train=X_train,
            y_train=y_train,
            params=params
        )

        print(f"5-Fold CV C-index : {cindex:.4f}")

        # Mise à jour des meilleurs hyperparamètres si le
        # C-index moyen est supérieur au meilleur obtenu.

        if cindex > best_cindex:
            best_cindex = cindex
            best_params = params.copy()

            print(" Nouveau meilleur modèle trouvé !")
            print(f"C-index : {best_cindex:.4f}")

    # Sauvegarde de la meilleure combinaison pour le modèle.

    best_models[model_name] = {
        "params": best_params,
        "cindex": best_cindex
    }

    # Affichage des meilleurs hyperparamètres retenus.

    print("\nBest parameters:")
    print(best_params)
    print(f"Best 5-Fold CV C-index : {best_cindex:.4f}")

In [ ]:
# ==========================================================
# 5. SAUVEGARDE DES MEILLEURS HYPERPARAMÈTRES
# ==========================================================


# Enregistrement des meilleurs hyperparamètres obtenus pour
# chaque modèle afin de pouvoir les réutiliser sans relancer
# la recherche sur grille.

with open("best_rsf_hyperparameters.json", "w") as f:
    json.dump(best_models, f, indent=4)

print("Meilleurs hyperparamètres sauvegardés dans 'best_rsf_hyperparameters.json'.")


In [ ]:
# ==========================================================
# RECHARGEMENT DES HYPERPARAMÈTRES (OPTIONNEL)
# ==========================================================

# Permet de recharger les meilleurs hyperparamètres
# précédemment sauvegardés.

with open("best_rsf_hyperparameters.json", "r") as f:
    best_models = json.load(f)

print("\nHyperparamètres rechargés :")

for model_name, infos in best_models.items():
    print(f"\n{model_name}")
    print(infos["params"])
    print(f"C-index CV : {infos['cindex']:.4f}")

In [ ]:
# ==========================================================
# ENTRAÎNEMENT DU MODÈLE FINAL SUR LE JEU DE DÉVELOPPEMENT
# ==========================================================

# Dictionnaire contenant les modèles finaux entraînés.

final_rsf_models = {}

for model_name, (X_train, X_internal_test) in models_dict.items():

    # Récupération des meilleurs hyperparamètres obtenus
    # par validation croisée.

    params = best_models[model_name]["params"]

    # Initialisation du Random Survival Forest.

    rsf = RandomSurvivalForest(
        n_estimators=params["n_estimators"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        random_state=42,
        n_jobs=-1
    )

    # Entraînement.

    rsf.fit(X_train, y_train)

    # Sauvegarde du modèle entraîné.

    final_rsf_models[model_name] = rsf

print("Final RSF models trained.")

In [ ]:
# ==========================================================
# ÉVALUATION DES MODÈLES SUR LE JEU DE VALIDATION INTERNE
# ==========================================================

# Dictionnaire contenant les performances des modèles
# sur le jeu de validation interne.

internal_results = {}

for model_name, (_, X_internal_test) in models_dict.items():

    # ======================================================
    # MODÈLE FINAL ENTRAÎNÉ
    # ======================================================

    model = final_rsf_models[model_name]

    # ======================================================
    # SCORES DE RISQUE
    # ======================================================

    risk_scores = model.predict(X_internal_test)

    # ======================================================
    # HARRELL C-INDEX
    # ======================================================

    harrell_cindex = concordance_index_censored(
        event_indicator=y_internal_test["event_irt"],
        event_time=y_internal_test["time_to_event"],
        estimate=risk_scores
    )[0]

    # ======================================================
    # UNO C-INDEX
    # ======================================================

    uno_cindex = concordance_index_ipcw(
        y_train,
        y_internal_test,
        risk_scores
    )[0]

    # ======================================================
    # ENREGISTREMENT DES PERFORMANCES
    # ======================================================

    internal_results[model_name] = {

        "Harrell C-index": harrell_cindex,

        "Uno C-index": uno_cindex,

    }

In [ ]:
# ==========================================================
# SYNTHÈSE DES PERFORMANCES SUR LE JEU DE VALIDATION INTERNE
# ==========================================================

# Conversion du dictionnaire de résultats en DataFrame
# pour faciliter la visualisation et l'analyse.

internal_results_df = (
    pd.DataFrame(internal_results)
      .T
      .round(4)
)

# Affichage des performances des modèles.

internal_results_df

In [ ]:
# ==========================================================
# FONCTION : PRÉDICTION DU TEMPS MÉDIAN DE SURVIE
# ==========================================================

def predict_median_survival(rsf, X):
    """
    Estime le temps médian de survie prédit à partir des
    fonctions de survie du Random Survival Forest.

    Une interpolation linéaire est utilisée lorsque la
    probabilité de survie franchit le seuil de 0.5 entre
    deux instants successifs.
    """

    # Prédiction des fonctions de survie

    surv_fns = rsf.predict_survival_function(X)

    # Initialisation du vecteur des temps médians prédits

    pred_median_survival = np.full(len(surv_fns), np.nan)

    # Parcours des fonctions de survie

    for i, fn in enumerate(surv_fns):

        idx = np.where(fn.y <= 0.5)[0]

        # La médiane n'est pas atteinte

        if len(idx) == 0:
            continue

        j = idx[0]

        # Premier point déjà inférieur à 0.5

        if j == 0:

            pred_median_survival[i] = fn.x[0]

        # Interpolation linéaire

        else:

            x1, x2 = fn.x[j - 1], fn.x[j]
            y1, y2 = fn.y[j - 1], fn.y[j]

            pred_median_survival[i] = np.interp(
                0.5,
                [y2, y1],
                [x2, x1]
            )

    return pred_median_survival

In [ ]:
# ==========================================================
# RESULTS
# ==========================================================

results_rsf = []

# ==========================================================
# PREDICTIONS
# ==========================================================

all_pred_rsf = []

# ==========================================================
# LOOP OVER THE MODELS
# ==========================================================

for model_name, (_, X_internal_test) in models_dict.items():

    print("\n")
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    # ======================================================
    # FINAL MODEL
    # ======================================================

    rsf = final_rsf_models[model_name]

    # ======================================================
    # C-INDEX SUR LE JEU D'ENTRAÎNEMENT
    # ======================================================

    cindex_train = rsf.score(
        models_dict[model_name][0],
        y_train
    )

    # ======================================================
    # SCORES DE RISQUE
    # ======================================================

    risk_scores = rsf.predict(
        X_internal_test
    )

    # ======================================================
    # HARRELL C-INDEX
    # ======================================================

    harrell_cindex = concordance_index_censored(
        event_indicator=y_internal_test["event_irt"],
        event_time=y_internal_test["time_to_event"],
        estimate=risk_scores
    )[0]

    # ======================================================
    # UNO C-INDEX
    # ======================================================

    uno_cindex = concordance_index_ipcw(
        y_train,
        y_internal_test,
        risk_scores
    )[0]

    # ======================================================
    # TEMPS MÉDIAN DE SURVIE PRÉDIT
    # ======================================================

    pred_median_survival = predict_median_survival(
        rsf,
        X_internal_test
    )

    # ======================================================
    # TABLEAU DES PRÉDICTIONS
    # ======================================================

    df_pred_rsf = pd.DataFrame({

        "patient_id": internal_test_df["patient_id"].values,
        "clinical_model": model_name,
        "time_observed": internal_test_df["time_to_event"].values,
        "event_irt": internal_test_df["event_irt"].values,
        "pred_median_survival": pred_median_survival,

    })

    # ======================================================
    # PATIENTS AYANT PRÉSENTÉ L'ÉVÉNEMENT
    # ET MÉDIANE ESTIMABLE
    # ======================================================

    df_pred_rsf = (
        df_pred_rsf
        .query("event_irt == 1")
        .dropna(subset=["pred_median_survival"])
        .copy()
    )

    # ======================================================
    # ERREURS DE PRÉDICTION
    # ======================================================

    df_pred_rsf["error_rsf"] = (
        df_pred_rsf["pred_median_survival"]
        - df_pred_rsf["time_observed"]
    )

    df_pred_rsf["abs_error_rsf"] = (
        df_pred_rsf["error_rsf"].abs()
    )

    # ======================================================
    # SAUVEGARDE DES PRÉDICTIONS
    # ======================================================

    all_pred_rsf.append(df_pred_rsf)

    # ======================================================
    # MÉTRIQUES
    # ======================================================

    mae = mean_absolute_error(
        df_pred_rsf["time_observed"],
        df_pred_rsf["pred_median_survival"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            df_pred_rsf["time_observed"],
            df_pred_rsf["pred_median_survival"]
        )
    )

    # ======================================================
    # AFFICHAGE
    # ======================================================

    print(f"C-index train      : {cindex_train:.4f}")
    print(f"Harrell C-index    : {harrell_cindex:.4f}")
    print(f"Uno C-index        : {uno_cindex:.4f}")
    print(f"Patients évalués   : {len(df_pred_rsf)}")
    print(f"MAE                : {mae:.2f} années")
    print(f"RMSE               : {rmse:.2f} années")

    # ======================================================
    # SAUVEGARDE DES RÉSULTATS
    # ======================================================

    results_rsf.append({

        "Model": model_name,
        "C-index train": round(cindex_train, 4),
        "Harrell C-index": round(harrell_cindex, 4),
        "Uno C-index": round(uno_cindex, 4),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2)

    })

# ==========================================================
# TABLEAU RÉCAPITULATIF
# ==========================================================

results_rsf_df = pd.DataFrame(results_rsf)

print("\n")
print("=" * 60)
print("RSF RESULTS — INTERNAL TEST SET")
print("=" * 60)

display(results_rsf_df)

# ==========================================================
# CONCATÉNATION DES PRÉDICTIONS
# ==========================================================

all_pred_rsf_df = pd.concat(
    all_pred_rsf,
    ignore_index=True
)

print("\n")
print("=" * 60)
print("RSF PREDICTIONS")
print("=" * 60)

display(all_pred_rsf_df.head())

print(f"\nShape : {all_pred_rsf_df.shape}")

#### 2. XGBOOST SURVIVAL AFT

In [ ]:
# ==========================================================
# 5-FOLD CROSS-VALIDATION
# XGBOOST SURVIVAL AFT
# Sélection des hyperparamètres par MAE
# ==========================================================

from sklearn.metrics import mean_absolute_error


def cross_validation_xgb_aft(
    X_train,
    y_train,
    params,
    n_splits=5
):

    # ======================================================
    # STRATIFIED K-FOLD
    # ======================================================

    cv = StratifiedKFold(

        n_splits=n_splits,

        shuffle=True,

        random_state=42

    )

    # ======================================================
    # INITIALISATION DES OBJETS DE STOCKAGE
    # ======================================================

    cindices = []

    maes = []

    best_iterations = []

    fold_results = []

    # ======================================================
    # BOUCLE SUR LES FOLDS
    # ======================================================

    for fold, (train_idx, val_idx) in enumerate(

        cv.split(
            X_train,
            y_train["event_irt"]
        ),

        start=1

    ):

        # ==================================================
        # TRAIN / VALIDATION
        # ==================================================

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train[train_idx]
        y_val = y_train[val_idx]

        # ==================================================
        # BORNES AFT - TRAIN
        # ==================================================

        y_lower_tr = y_tr["time_to_event"].astype(float).copy()

        y_upper_tr = y_tr["time_to_event"].astype(float).copy()

        y_upper_tr[
            y_tr["event_irt"] == 0
        ] = np.inf

        # ==================================================
        # BORNES AFT - VALIDATION
        # ==================================================

        y_lower_val = y_val["time_to_event"].astype(float).copy()

        y_upper_val = y_val["time_to_event"].astype(float).copy()

        y_upper_val[
            y_val["event_irt"] == 0
        ] = np.inf

        # ==================================================
        # DMATRIX
        # ==================================================

        dtrain = xgb.DMatrix(X_tr)

        dvalid = xgb.DMatrix(X_val)

        # ==================================================
        # LABELS TRAIN
        # ==================================================

        dtrain.set_float_info(
            "label_lower_bound",
            y_lower_tr
        )

        dtrain.set_float_info(
            "label_upper_bound",
            y_upper_tr
        )

        # ==================================================
        # LABELS VALIDATION
        # ==================================================

        dvalid.set_float_info(
            "label_lower_bound",
            y_lower_val
        )

        dvalid.set_float_info(
            "label_upper_bound",
            y_upper_val
        )

        # ==================================================
        # PARAMÈTRES XGBOOST
        # ==================================================

        xgb_params = {

            "objective": "survival:aft",

            "eval_metric": "aft-nloglik",

            "tree_method": "hist",

            "verbosity": 0,

            "nthread": -1,

            "seed": 42,

            **params

        }

        # ==================================================
        # ENTRAÎNEMENT
        # ==================================================

        model = xgb.train(

            params=xgb_params,

            dtrain=dtrain,

            num_boost_round=1000,

            evals=[
                (dvalid, "validation")
            ],

            early_stopping_rounds=60,

            verbose_eval=False

        )

        # ==================================================
        # PRÉDICTION
        # ==================================================

        pred_time = model.predict(dvalid)

        # ==================================================
        # HARRELL C-INDEX
        # ==================================================

        cindex = concordance_index_censored(

            y_val["event_irt"],

            y_val["time_to_event"],

            -pred_time

        )[0]

        cindices.append(cindex)

        # ==================================================
        # MAE (patients ayant présenté l'événement)
        # ==================================================

        event_mask = y_val["event_irt"] == 1

        mae = mean_absolute_error(

            y_val["time_to_event"][event_mask],

            pred_time[event_mask]

        )

        maes.append(mae)

        # ==================================================
        # BEST ITERATION
        # ==================================================

        best_iterations.append(

            model.best_iteration + 1

        )

        # ==================================================
        # RÉSULTATS DU FOLD
        # ==================================================

        fold_results.append({

            "fold": fold,

            "cindex": cindex,

            "mae": mae,

            "best_iteration": model.best_iteration + 1

        })

    # ======================================================
    # RETOUR
    # ======================================================

    return {

        "mean_cindex": np.mean(cindices),

        "std_cindex": np.std(cindices),

        "mean_mae": np.mean(maes),

        "std_mae": np.std(maes),

        "median_best_iteration": int(

            np.median(best_iterations)

        ),

        "fold_results": fold_results

    }

In [ ]:
# ==========================================================
# JEUX DE DONNÉES ASSOCIÉS AUX MODÈLES CLINIQUES
# ==========================================================

# Association de chaque modèle clinique avec son jeu de
# développement et son jeu de validation interne.

MODELS = {

    "Baseline": {
        "X_train": X_train_model1,
        "X_test": X_internal_model1,
    },

    "Before_35": {
        "X_train": X_train_model2,
        "X_test": X_internal_model2,
    },

    "Combined": {
        "X_train": X_train_model3,
        "X_test": X_internal_model3,
    }

}

In [ ]:
# ==========================================================
# RECHERCHE DES HYPERPARAMÈTRES PAR OPTUNA
# ==========================================================

# ==========================================================
# FONCTION OBJECTIF
# ==========================================================

def objective(trial, X_train, y_train):

    params = {

        # --------------------------------------------------
        # Paramètres principaux
        # --------------------------------------------------

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.003,
            0.10,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            6
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            20
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            10.0
        ),

        # --------------------------------------------------
        # Sampling
        # --------------------------------------------------

        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.50,
            1.00
        ),

        # --------------------------------------------------
        # Régularisation
        # --------------------------------------------------

        "lambda": trial.suggest_float(
            "lambda",
            1e-3,
            20,
            log=True
        ),

        "alpha": trial.suggest_float(
            "alpha",
            1e-3,
            20,
            log=True
        ),

        # --------------------------------------------------
        # Paramètres spécifiques à l'AFT
        # --------------------------------------------------

        "aft_loss_distribution": trial.suggest_categorical(
            "aft_loss_distribution",
            [
                "normal",
                "logistic",
                "extreme"
            ]
        ),

        "aft_loss_distribution_scale": trial.suggest_float(
            "aft_loss_distribution_scale",
            0.30,
            1.20
        ),

        # --------------------------------------------------
        # Paramètres Hist
        # --------------------------------------------------

        "max_bin": trial.suggest_categorical(
            "max_bin",
            [
                128,
                256,
                512
            ]
        ),

        "grow_policy": trial.suggest_categorical(
            "grow_policy",
            [
                "depthwise",
                "lossguide"
            ]
        ),

        "max_delta_step": trial.suggest_float(
            "max_delta_step",
            0.0,
            5.0
        )

    }

    cv_results = cross_validation_xgb_aft(

        X_train=X_train,

        y_train=y_train,

        params=params,

        n_splits=5

    )

    trial.set_user_attr(

        "best_iteration",

        cv_results["median_best_iteration"]

    )

    return cv_results["mean_mae"]


# ==========================================================
# OPTIMISATION DES TROIS MODÈLES
# ==========================================================

optuna_results = {}

for model_name, data in MODELS.items():

    print("\n" + "=" * 70)
    print(f"OPTUNA - {model_name}")
    print("=" * 70)

    X_train = data["X_train"]

    sampler = optuna.samplers.TPESampler(
        seed=42,
        multivariate=True
    )

    study = optuna.create_study(

        direction="minimize",

        sampler=sampler

    )

    study.optimize(

        lambda trial: objective(
            trial,
            X_train,
            y_train
        ),

        n_trials=50,

        show_progress_bar=True

    )

    optuna_results[model_name] = {

        "best_params": study.best_trial.params,

        "best_iteration": study.best_trial.user_attrs[
            "best_iteration"
        ],

        "best_cv": study.best_value

    }

    print("\nMeilleur Harrell CV :", study.best_value)

    print("Best iteration :", study.best_trial.user_attrs[
        "best_iteration"
    ])

    print("\nBest parameters :")

    for key, value in study.best_trial.params.items():
        print(f"{key:<30} : {value}")

print("\nOptimisation terminée.")

In [ ]:
# ==========================================================
# SAUVEGARDE DES MEILLEURS HYPERPARAMÈTRES
# ==========================================================

# Enregistrement des meilleurs hyperparamètres obtenus
# pour chacun des trois modèles cliniques.

with open("best_xgb_aft_hyperparameters.json", "w") as f:
    json.dump(optuna_results, f, indent=4)

print(
    "Meilleurs hyperparamètres sauvegardés dans "
    "'best_xgb_aft_hyperparameters.json'."
)

In [ ]:
# ==========================================================
# RECHARGEMENT DES HYPERPARAMÈTRES (OPTIONNEL)
# ==========================================================

# Permet de recharger les meilleurs hyperparamètres
# précédemment sauvegardés.

with open("best_xgb_aft_hyperparameters.json", "r") as f:
    optuna_results = json.load(f)

In [ ]:
# ==========================================================
# ENTRAÎNEMENT DES MODÈLES FINAUX
# ==========================================================


# Dictionnaire contenant les modèles finaux entraînés.
final_xgb_models = {}

for model_name, data in MODELS.items():

    print("\n" + "=" * 60)
    print(f"ENTRAÎNEMENT FINAL - {model_name}")
    print("=" * 60)

    X_train = data["X_train"]

    # ======================================================
    # CONSTRUCTION DES BORNES AFT
    # ======================================================

    y_lower = y_train["time_to_event"].astype(float).copy()

    y_upper = y_train["time_to_event"].astype(float).copy()

    y_upper[y_train["event_irt"] == 0] = np.inf

    # ======================================================
    # CRÉATION DE LA DMATRIX D'ENTRAÎNEMENT
    # ======================================================

    dtrain = xgb.DMatrix(X_train)

    dtrain.set_float_info(
        "label_lower_bound",
        y_lower
    )

    dtrain.set_float_info(
        "label_upper_bound",
        y_upper
    )

    # ======================================================
    # RÉCUPÉRATION DES MEILLEURS HYPERPARAMÈTRES
    # ======================================================

    best_params = optuna_results[model_name]["best_params"]

    best_num_boost_round = optuna_results[model_name]["best_iteration"]

    # ======================================================
    # PARAMÈTRES DU MODÈLE FINAL
    # ======================================================

    final_params = {

        "objective": "survival:aft",

        "eval_metric": "aft-nloglik",

        "tree_method": "hist",

        "verbosity": 0,

        "nthread": -1,

        "seed": 42,

        **best_params

    }

    # ======================================================
    # ENTRAÎNEMENT DU MODÈLE FINAL SUR L'ENSEMBLE
    # DU JEU DE DÉVELOPPEMENT
    # ======================================================

    final_model = xgb.train(

        params=final_params,

        dtrain=dtrain,

        num_boost_round=best_num_boost_round

    )

    # Sauvegarde du modèle entraîné.

    final_xgb_models[model_name] = final_model

    print(f"Nombre d'arbres : {best_num_boost_round}")

print("\nTous les modèles ont été entraînés.")

In [ ]:
# ==========================================================
# PRÉDICTIONS DES MODÈLES SUR LE JEU DE VALIDATION INTERNE
# ==========================================================

# Dictionnaire contenant les prédictions de chaque modèle
# clinique sur le jeu de validation interne.
predictions = {}

for model_name, data in MODELS.items():

    print("\n" + "=" * 60)
    print(f"PRÉDICTIONS - {model_name}")
    print("=" * 60)

    X_internal_test = data["X_test"]

    # ======================================================
    # CRÉATION DE LA DMATRIX DE VALIDATION
    # ======================================================

    dtest = xgb.DMatrix(X_internal_test)

    # ======================================================
    # PRÉDICTION DU TEMPS JUSQU'À L'IRT
    # ======================================================

    predicted_time_to_event = final_xgb_models[model_name].predict(dtest)

    # ======================================================
    # RASSEMBLEMENT DES PRÉDICTIONS
    # ======================================================

    predictions_df = pd.DataFrame({

        "event_irt": y_internal_test["event_irt"],

        "true_time_to_event": y_internal_test["time_to_event"],

        "predicted_time_to_event": predicted_time_to_event

    })

    # Sauvegarde des prédictions.
    predictions[model_name] = predictions_df

    # Affichage d'un aperçu des prédictions.
    print(predictions_df.head())

In [ ]:
# ==========================================================
# ÉVALUATION DES MODÈLES XGBOOST SURVIVAL AFT
# SUR LE JEU DE VALIDATION INTERNE
# ==========================================================

# Liste contenant les performances des différents modèles.
results = []

for model_name, data in MODELS.items():

    # Récupération des prédictions du modèle.

    predictions_df = predictions[model_name]

    predicted_time_to_event = predictions_df[
        "predicted_time_to_event"
    ].to_numpy()

    # ======================================================
    # CALCUL DU C-INDEX DE HARRELL
    # ======================================================

    harrell_cindex = concordance_index_censored(

        event_indicator=y_internal_test["event_irt"],

        event_time=y_internal_test["time_to_event"],

        estimate=-predicted_time_to_event

    )[0]

    # ======================================================
    # CALCUL DU C-INDEX D'UNO
    # ======================================================

    uno_cindex = concordance_index_ipcw(

        survival_train=y_train,

        survival_test=y_internal_test,

        estimate=-predicted_time_to_event

    )[0]

    # ======================================================
    # CALCUL DES ERREURS DE PRÉDICTION
    # (PATIENTS AYANT PRÉSENTÉ L'ÉVÉNEMENT)
    # ======================================================

    event_mask = y_internal_test["event_irt"]

    true_time_events = y_internal_test["time_to_event"][
        event_mask
    ]

    predicted_time_events = predicted_time_to_event[
        event_mask
    ]

    mae = mean_absolute_error(

        true_time_events,

        predicted_time_events

    )

    rmse = np.sqrt(

        mean_squared_error(

            true_time_events,

            predicted_time_events

        )

    )

    # ======================================================
    # ENREGISTREMENT DES PERFORMANCES
    # ======================================================

    results.append({

        "Model": model_name,

        "Harrell C-index": harrell_cindex,

        "Uno C-index": uno_cindex,

        "MAE (years)": mae,

        "RMSE (years)": rmse

    })

# ==========================================================
# TABLEAU RÉCAPITULATIF DES PERFORMANCES
# ==========================================================

results_df = (
    pd.DataFrame(results)
    .sort_values("Harrell C-index", ascending=False)
    .reset_index(drop=True)
)

print("=" * 80)
print("RÉSULTATS DES MODÈLES XGBOOST SURVIVAL AFT")
print("=" * 80)

print(results_df.round(4))

results_df

### Analyse sur les prédictions 

In [ ]:
# ==========================================================
# SAUVEGARDE HTML — TABLEAUX ML
# ==========================================================


# ----------------------------------------------------------
# Dossier de sortie
# ----------------------------------------------------------

output_dir = (
    r".\tables\survival_models"
)

os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------------------------
# Chemin du fichier HTML
# ----------------------------------------------------------

html_path = os.path.join(
    output_dir,
    "survival_ml_results_ckd_epi.html"
)

# ----------------------------------------------------------
# Génération du HTML
# ----------------------------------------------------------

html_content = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>Survival Machine Learning Results</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 40px;
    background-color: white;
}}

h1 {{
    color: #222222;
}}

h2 {{
    margin-top: 50px;
    color: #333333;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 15px;
    margin-bottom: 40px;
}}

th,
td {{
    border: 1px solid #cccccc;
    padding: 8px;
    text-align: center;
}}

th {{
    background-color: #f2f2f2;
}}

tr:nth-child(even) {{
    background-color: #fafafa;
}}

</style>

</head>

<body>

<h1>Comparison of Survival Machine Learning Models</h1>

<h2>Random Survival Forest</h2>

{results_rsf_df.to_html(index=False)}

<h2>XGBoost Survival AFT</h2>

{results_df.to_html(index=False)}

</body>

</html>
"""

# ----------------------------------------------------------
# Sauvegarde
# ----------------------------------------------------------

with open(
    html_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(html_content)

print(f"HTML sauvegardé :\n{html_path}")

### Analyse des erreurs

#### 1. XGBoost AFT

In [ ]:
# ==========================================================
# DISTRIBUTION DES ERREURS DE PRÉDICTION
# TEMPS RESTANT JUSQU'À L'IRT
# ==========================================================



# ----------------------------------------------------------
# Modèle à analyser
# ----------------------------------------------------------

MODEL = "Combined"

predictions_df = predictions[MODEL].copy()

# ----------------------------------------------------------
# Patients ayant présenté l'événement
# ----------------------------------------------------------

events_df = predictions_df[
    predictions_df["event_irt"]
].copy()

# ----------------------------------------------------------
# Calcul des erreurs
# ----------------------------------------------------------

events_df["prediction_error"] = (

    events_df["predicted_time_to_event"]

    - events_df["true_time_to_event"]

)

# ----------------------------------------------------------
# Statistiques descriptives
# ----------------------------------------------------------

stats = pd.DataFrame({

    "Statistic": [

        "Number of patients",

        "Mean error",

        "Median error",

        "Standard deviation",

        "Minimum",

        "Q1",

        "Q3",

        "Maximum"

    ],

    "Value": [

        len(events_df),

        events_df["prediction_error"].mean(),

        events_df["prediction_error"].median(),

        events_df["prediction_error"].std(),

        events_df["prediction_error"].min(),

        events_df["prediction_error"].quantile(0.25),

        events_df["prediction_error"].quantile(0.75),

        events_df["prediction_error"].max()

    ]

})

print(stats.round(3))

# ----------------------------------------------------------
# Histogramme des erreurs
# ----------------------------------------------------------

fig = px.histogram(

    events_df,

    x="prediction_error",

    nbins=40,

    title=(
        f"Prediction error distribution - {MODEL}"
    ),

    labels={

        "prediction_error":
        "Prediction error (predicted - observed years)"

    }

)

fig.add_vline(

    x=0,

    line_dash="dash",

    line_color="red"

)

fig.update_layout(

    template="plotly_white",

    xaxis_title="Prediction error (years)",

    yaxis_title="Number of patients"

)

fig.show()

**Interprétation :**

  La distribution des erreurs de prédiction montre que les erreurs sont principalement concentrées autour de zéro, indiquant une précision globalement satisfaisante du modèle. Cependant, la distribution est légèrement décalée vers les valeurs positives, suggérant une tendance du modèle à surestimer le temps restant jusqu'à l'insuffisance rénale terminale. Quelques observations présentent des erreurs importantes, aussi bien en surestimation qu'en sous-estimation, traduisant la présence de patients atypiques pour lesquels la prédiction est plus difficile.

In [ ]:
# ==========================================================
# TEMPS RESTANT OBSERVÉ VS TEMPS RESTANT PRÉDIT
# ==========================================================


# ----------------------------------------------------------
# Modèle à analyser
# ----------------------------------------------------------

MODEL = "Combined"

predictions_df = predictions[MODEL].copy()

# ----------------------------------------------------------
# Patients ayant présenté l'événement
# ----------------------------------------------------------

events_df = predictions_df[
    predictions_df["event_irt"]
].copy()

# ----------------------------------------------------------
# Valeurs observées et prédites
# ----------------------------------------------------------

x = events_df["true_time_to_event"]

y = events_df["predicted_time_to_event"]

# ----------------------------------------------------------
# Limites du graphique
# ----------------------------------------------------------

min_value = min(x.min(), y.min())

max_value = max(x.max(), y.max())

# ----------------------------------------------------------
# Nuage de points
# ----------------------------------------------------------

fig = px.scatter(

    events_df,

    x="true_time_to_event",

    y="predicted_time_to_event",

    opacity=0.75,

    title=(
        f"Observed vs predicted time to IRT - {MODEL}"
    ),

    labels={

        "true_time_to_event":
        "Observed time to IRT (years)",

        "predicted_time_to_event":
        "Predicted time to IRT (years)"

    }

)

# ----------------------------------------------------------
# Droite idéale y = x
# ----------------------------------------------------------

fig.add_shape(

    type="line",

    x0=min_value,

    y0=min_value,

    x1=max_value,

    y1=max_value,

    line=dict(

        color="red",

        dash="dash"

    )

)

fig.update_layout(

    template="plotly_white"

)

fig.show()

In [ ]:
# ==========================================================
# ERREUR DE PRÉDICTION EN FONCTION DU DFG
# ==========================================================


# ----------------------------------------------------------
# Modèle analysé
# ----------------------------------------------------------

MODEL = "Combined"

predictions_df = predictions[MODEL].copy()

# ----------------------------------------------------------
# Patients avec événement observé
# ----------------------------------------------------------

events_df = predictions_df[
    predictions_df["event_irt"]
].copy()

# ----------------------------------------------------------
# Ajout du DFG
# ----------------------------------------------------------

events_df["ckd_epi"] = (

    MODELS[MODEL]["X_test"]

    .loc[events_df.index, "ckd_epi"]

)

# ----------------------------------------------------------
# Calcul de l'erreur
# ----------------------------------------------------------

events_df["prediction_error"] = (

    events_df["predicted_time_to_event"]

    - events_df["true_time_to_event"]

)

# ----------------------------------------------------------
# Nuage de points
# ----------------------------------------------------------

fig = px.scatter(

    events_df,

    x="ckd_epi",

    y="prediction_error",

    opacity=0.7,

    trendline="lowess",

    title="Prediction error according to baseline eGFR",

    labels={

        "ckd_epi":"Baseline eGFR",

        "prediction_error":"Prediction error (years)"

    }

)

fig.add_hline(

    y=0,

    line_dash="dash",

    line_color="red"

)

fig.update_layout(

    template="plotly_white"

)

fig.show()

In [ ]:
# ==========================================================
# MAE SELON LES STADES CKD
# ==========================================================


# ----------------------------------------------------------
# Définition des stades CKD
# ----------------------------------------------------------

events_df["CKD stage"] = np.select(

    [

        events_df["ckd_epi"] >= 90,

        (events_df["ckd_epi"] >= 60) &
        (events_df["ckd_epi"] < 90),

        (events_df["ckd_epi"] >= 30) &
        (events_df["ckd_epi"] < 60),

    ],

    [

        "G1 (≥90)",

        "G2 (60–89)",

        "G3 (30–59)",

    ],

    default=""

)

# ----------------------------------------------------------
# Tableau des performances
# ----------------------------------------------------------

summary = (

    events_df

    .groupby("CKD stage")

    .agg(

        n=("prediction_error", "size"),

        mean_error=("prediction_error", "mean"),

        median_error=("prediction_error", "median"),

    )

)

summary["MAE"] = (

    events_df

    .groupby("CKD stage")

    .apply(
        lambda x: np.abs(
            x["prediction_error"]
        ).mean(),
        include_groups=False
    )

)

summary["RMSE"] = (

    events_df

    .groupby("CKD stage")

    .apply(
        lambda x: np.sqrt(
            np.mean(
                x["prediction_error"] ** 2
            )
        ),
        include_groups=False
    )

)

summary = summary.reset_index()

print(summary)

**Interprétation clinique :**

Ce résultat est cohérent avec l'évolution naturelle de la PKRAD.

Les patients G1 sont à un stade précoce de la maladie. Leur délai avant l'IRT est souvent long et influencé par de nombreux facteurs (génétiques, vitesse de décroissance du DFG, hypertension, etc.). Cette variabilité rend la prédiction plus difficile.
Les patients G3 sont plus proches de l'IRT. Leur trajectoire est généralement plus avancée et le temps restant est plus facile à estimer, ce qui explique des erreurs plus faibles et un biais limité.

In [ ]:
# ==========================================================
# RECONSTRUCTION DU GÉNOTYPE
# ==========================================================

genetic_columns = [

    "genetic_GUR",
    "genetic_PKD1_non_truncating",
    "genetic_PKD1_truncating",
    "genetic_PKD2",
    "genetic_other",

]

# ==========================================================
# DISTRIBUTION DES GÉNOTYPES DANS LE TRAIN ET LE TEST INTERNE
# ==========================================================

# ----------------------------------------------------------
# Distribution dans le jeu d'entraînement
# ----------------------------------------------------------

train_counts = (
    train_df["genetic"]
    .value_counts()
    .rename("Train")
)

# ----------------------------------------------------------
# Distribution dans le jeu de test interne
# ----------------------------------------------------------

internal_test_counts = (
    internal_test_df["genetic"]
    .value_counts()
    .rename("Internal test")
)

# ==========================================================
# DISTRIBUTION DES GÉNOTYPES DANS LE TRAIN, LE TEST ET
# LES PATIENTS AVEC ÉVÉNEMENT
# ==========================================================


# ----------------------------------------------------------
# Distribution dans le jeu d'entraînement
# ----------------------------------------------------------

train_counts = (
    train_df["genetic"]
    .value_counts()
    .rename("Train")
)

# ----------------------------------------------------------
# Distribution dans le jeu de test interne
# ----------------------------------------------------------

internal_test_counts = (
    internal_test_df["genetic"]
    .value_counts()
    .rename("Internal test")
)

# ----------------------------------------------------------
# Ajout du génotype aux patients évalués
# ----------------------------------------------------------

events_df["genetic"] = (
    internal_test_df.loc[events_df.index, "genetic"].values
)

# ----------------------------------------------------------
# Distribution des patients avec événement
# (on utilise les mêmes index que events_df)
# ----------------------------------------------------------

internal_test_event_counts = (
    internal_test_df
    .loc[events_df.index, "genetic"]
    .value_counts()
    .rename("Internal test (events)")
)

# ----------------------------------------------------------
# Tableau récapitulatif
# ----------------------------------------------------------

distribution = (
    pd.concat(
        [
            train_counts,
            internal_test_counts,
            internal_test_event_counts,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)

# ----------------------------------------------------------
# Pourcentages
# ----------------------------------------------------------

distribution["Train (%)"] = (
    100 * distribution["Train"] / distribution["Train"].sum()
).round(1)

distribution["Internal test (%)"] = (
    100 * distribution["Internal test"] / distribution["Internal test"].sum()
).round(1)

distribution["Events (%)"] = (
    100
    * distribution["Internal test (events)"]
    / distribution["Internal test (events)"].sum()
).round(1)

print(distribution)

In [ ]:
# ==========================================================
# PERFORMANCES PAR GÉNOTYPE
# ==========================================================


# ----------------------------------------------------------
# Nombre de patients par génotype dans le jeu d'entraînement
# ----------------------------------------------------------

train_counts = (
    train_df
    .groupby("genetic")
    .size()
    .rename("n_train")
)

# ----------------------------------------------------------
# Nombre de patients avec événement dans le jeu d'entraînement
# ----------------------------------------------------------

train_event_counts = (

    train_df

    .loc[train_df["event_irt"] == 1]

    .groupby("genetic")

    .size()

    .rename("n_train_events")

)

# ----------------------------------------------------------
# Résumé sur les patients évalués (événements du test)
# ----------------------------------------------------------

results_genetic = (

    events_df

    .groupby("genetic")

    .agg(

        n_test_events=("prediction_error", "size"),

        mean_error=("prediction_error", "mean"),

        median_error=("prediction_error", "median"),

    )

)

# ----------------------------------------------------------
# Ajout des effectifs du train
# ----------------------------------------------------------

results_genetic = (
    results_genetic
    .join(train_counts)
    .join(train_event_counts)
)

# ----------------------------------------------------------
# Calcul du MAE et du RMSE
# ----------------------------------------------------------

mae = []
rmse = []

for genetic, group in events_df.groupby("genetic"):

    mae.append(

        mean_absolute_error(

            group["true_time_to_event"],
            group["predicted_time_to_event"]

        )

    )

    rmse.append(

        np.sqrt(

            mean_squared_error(

                group["true_time_to_event"],
                group["predicted_time_to_event"]

            )

        )

    )

results_genetic["MAE"] = mae
results_genetic["RMSE"] = rmse

# ----------------------------------------------------------
# Ordre des colonnes
# ----------------------------------------------------------

results_genetic = (

    results_genetic

    .reset_index()

    [[
        "genetic",
        "n_train",
        "n_train_events",
        "n_test_events",
        "MAE",
        "RMSE",
        "mean_error",
        "median_error",
    ]]

    .sort_values(
        by="n_train",
        ascending=False,
    )

)

print(results_genetic)

In [ ]:
# ==========================================================
# BOXPLOT DES ERREURS PAR GÉNOTYPE
# ==========================================================

fig = px.box(

    events_df,

    x="genetic",

    y="prediction_error",

    points="all",

    title="Prediction error according to genetic group",

    labels={

        "genetic":"Genetic group",

        "prediction_error":"Prediction error (years)"

    }

)

fig.add_hline(

    y=0,

    line_dash="dash",

    line_color="red"

)

fig.update_layout(

    template="plotly_white"

)

fig.show()

In [ ]:
# ==========================================================
# PERFORMANCES SELON LE TEMPS RÉEL RESTANT AVANT L'IRT
# ==========================================================

# ----------------------------------------------------------
# Catégories de temps restant (train)
# ----------------------------------------------------------

train_df["remaining_time_group"] = pd.cut(
    train_df["time_to_event"],
    bins=[0, 5, 10, 15, np.inf],
    labels=[
        "<5 years",
        "5–10 years",
        "10–15 years",
        ">15 years",
    ],
    right=False,
)

# ----------------------------------------------------------
# Catégories de temps restant (test)
# ----------------------------------------------------------

events_df["remaining_time_group"] = pd.cut(
    events_df["true_time_to_event"],
    bins=[0, 5, 10, 15, np.inf],
    labels=[
        "<5 years",
        "5–10 years",
        "10–15 years",
        ">15 years",
    ],
    right=False,
)

# ----------------------------------------------------------
# Effectifs train
# ----------------------------------------------------------

train_counts = (
    train_df
    .groupby("remaining_time_group", observed=True)
    .agg(
        n_train=("event_irt", "size"),
        n_events_train=("event_irt", "sum"),
    )
)

# ----------------------------------------------------------
# Résultats test
# ----------------------------------------------------------

test_counts = (
    events_df
    .groupby("remaining_time_group", observed=True)
    .agg(
        n_events_test=("event_irt", "sum"),
        mean_error=("prediction_error", "mean"),
        median_error=("prediction_error", "median"),
    )
)

# ----------------------------------------------------------
# MAE / RMSE
# ----------------------------------------------------------

metrics = []

for group_name, group in events_df.groupby("remaining_time_group", observed=True):

    metrics.append(
        {
            "remaining_time_group": group_name,
            "MAE": mean_absolute_error(
                group["true_time_to_event"],
                group["predicted_time_to_event"],
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    group["true_time_to_event"],
                    group["predicted_time_to_event"],
                )
            ),
        }
    )

metrics = (
    pd.DataFrame(metrics)
    .set_index("remaining_time_group")
)

# ----------------------------------------------------------
# Tableau final
# ----------------------------------------------------------

results_remaining = (
    train_counts
    .join(test_counts)
    .join(metrics)
    .reset_index()
)

print(results_remaining)

In [ ]:
# ==========================================================
# BOXPLOT DES ERREURS SELON LE TEMPS RESTANT
# ==========================================================


fig = px.box(

    events_df,

    x="remaining_time_group",

    y="prediction_error",

    points="all",

    color="remaining_time_group",

    labels={

        "remaining_time_group":"Remaining time",

        "prediction_error":"Prediction error (years)"

    },

    title="Prediction error according to the true remaining time"

)

fig.add_hline(

    y=0,

    line_dash="dash",

    line_color="red"

)

fig.update_layout(

    template="plotly_white",

    showlegend=False,

)

fig.show()

In [ ]:
# ==========================================================
# KRUSKAL-WALLIS
# ==========================================================


groups = [

    g["prediction_error"].values

    for _, g in events_df.groupby(

        "remaining_time_group",

        observed=True

    )

]

stat, p = kruskal(*groups)

print(f"Kruskal statistic : {stat:.3f}")

print(f"p-value           : {p:.4f}")

**Interprétation :**

  Les performances du modèle varient en fonction du temps restant avant l'insuffisance rénale terminale (IRT). Les prédictions sont les plus précises chez les patients proches de l'IRT, avec un MAE de 2,73 ans lorsque le temps restant est inférieur à 5 ans. À mesure que l'horizon de prédiction s'allonge, la précision diminue progressivement : le MAE passe à 3,70 ans pour un temps restant compris entre 5 et 10 ans, puis à 4,56 ans entre 10 et 15 ans, pour atteindre 6,14 ans lorsque l'IRT est attendue dans plus de 15 ans. Cette tendance est également observée avec le RMSE.

Ces résultats sont cohérents avec l'évolution de la maladie. Chez les patients proches de l'IRT, la dégradation de la fonction rénale est généralement plus marquée et plus régulière, ce qui facilite la prédiction. En revanche, chez les patients encore éloignés de l'IRT, l'évolution est plus variable d'un individu à l'autre, rendant les prédictions plus difficiles.

Le test de Kruskal–Wallis ne met toutefois pas en évidence de différence statistiquement significative entre les groupes (p = 0,063). Ainsi, même si une tendance à une diminution des performances lorsque le temps restant avant l'IRT augmente est observée, celle-ci ne peut pas être confirmée statistiquement dans notre échantillon.

In [ ]:
# ==========================================================
# ANALYSE DES 20 PLUS GROSSES ERREURS
# ==========================================================

# ----------------------------------------------------------
# Top 20 erreurs absolues
# ----------------------------------------------------------

top_errors = (

    events_df

    .assign(abs_error=lambda x: x["prediction_error"].abs())

    .sort_values("abs_error", ascending=False)

    .head(20)

)

# ----------------------------------------------------------
# Variables explicatives correspondantes
# ----------------------------------------------------------

patient_info = MODELS[MODEL]["X_test"].loc[top_errors.index].copy()

# Reconstruction du sexe
patient_info["sex"] = (

    patient_info[
        [
            "sex_Female",
            "sex_Male",
        ]
    ]

    .idxmax(axis=1)

    .str.replace("sex_", "", regex=False)

)

# Reconstruction du génotype
patient_info["genetic"] = (

    patient_info[
        [
            "genetic_GUR",
            "genetic_PKD1_non_truncating",
            "genetic_PKD1_truncating",
            "genetic_PKD2",
            "genetic_other",
        ]
    ]

    .idxmax(axis=1)

    .str.replace("genetic_", "", regex=False)

)

# ----------------------------------------------------------
# Tableau final
# ----------------------------------------------------------

top_errors_table = pd.DataFrame({

    "True time": top_errors["true_time_to_event"],
    "Predicted time": top_errors["predicted_time_to_event"],
    "Error": top_errors["prediction_error"],
    "Absolute error": top_errors["abs_error"],

    "Age": patient_info["creat_age"],
    "eGFR": patient_info["ckd_epi"],

    "Sex": patient_info["sex"],
    "Genetic": patient_info["genetic"],

    "Hypertension": patient_info["has_hypertension_baseline"],
    "Urologic event": patient_info["has_urologic_event_baseline"],

})

display(top_errors_table)

In [ ]:
# ==========================================================
# RÉSUMÉ DES PLUS GROSSES ERREURS
# ==========================================================

print(f"Erreur absolue moyenne : {top_errors['abs_error'].mean():.2f} ans")
print(f"Erreur absolue médiane : {top_errors['abs_error'].median():.2f} ans")

print("\nGénotypes")
print(patient_info["genetic"].value_counts())

print("\nSexe")
print(patient_info["sex"].value_counts())

print("\nHypertension")
print(patient_info["has_hypertension_baseline"].value_counts())

print("\nÉvènement urologique")
print(patient_info["has_urologic_event_baseline"].value_counts())

**Interprétation :**

  Les 20 plus grandes erreurs de prédiction présentent une erreur absolue moyenne de 11,83 ans (médiane : 10,73 ans), soit des erreurs nettement supérieures à la performance moyenne du modèle.

Parmi ces patients, 11 sur 20 sont porteurs d'une mutation PKD1 truncating, tandis que les autres génotypes sont représentés en plus faible nombre. Cette observation est toutefois à interpréter avec prudence, car PKD1 truncating est également le génotype le plus fréquent dans la cohorte, ce qui augmente naturellement sa représentation parmi les plus grandes erreurs.

La répartition selon le sexe est relativement équilibrée (11 femmes et 9 hommes), ce qui ne suggère pas de différence marquée entre les deux sexes.

De même, les statuts d'hypertension et d'événements urologiques ne montrent pas de profil particulier parmi ces patients. Les plus grandes erreurs semblent donc être réparties entre différents profils cliniques plutôt que concentrées dans un sous-groupe spécifique.

In [ ]:
# ==========================================================
# TOP 20 DES PLUS GROSSES ERREURS
# ==========================================================

fig = px.bar(

    top_errors_table,

    x=top_errors_table.index.astype(str),

    y="Absolute error",

    color="Genetic",

    hover_data=[

        "True time",
        "Predicted time",
        "Error",
        "eGFR",
        "Age",
        "Sex",
        "Hypertension",
        "Urologic event",

    ],

    labels={

        "x": "Patient",
        "Absolute error": "Absolute error (years)",

    },

    title="Top 20 largest prediction errors",

)

fig.update_layout(

    template="plotly_white",

    xaxis_title="Patient",
    yaxis_title="Absolute error (years)",

)

fig.show()

**Interprétation :**

  Les plus fortes erreurs de prédiction sont observées dans différents profils génétiques, sans qu'un génotype particulier ne soit systématiquement associé aux erreurs les plus importantes. La majorité des patients les plus mal prédits appartiennent toutefois au groupe PKD1 tronquant, qui est également le plus représenté dans la cohorte. Ces résultats suggèrent que les erreurs importantes ne semblent pas être liées à une seule catégorie génétique mais reflètent plutôt la variabilité individuelle des patients.

In [ ]:
# ==========================================================
# SHAP VALUES
# ==========================================================


# ==========================================================
# MODÈLE À EXPLIQUER
# ==========================================================

MODEL = "Combined"

X_test = MODELS["Combined"]["X_test"]

explainer = shap.TreeExplainer(
    final_xgb_models["Combined"]
)

shap_values = explainer.shap_values(X_test)

In [ ]:
# Importance globale
shap.summary_plot(
    shap_values,
    X_test,
    plot_type="bar"
)

In [ ]:
# ==========================================================
# SHAP SUMMARY PLOT
# ==========================================================

shap.summary_plot(
    shap_values,
    X_test
)

In [ ]:
# Tableau des importances moyennes


importance = pd.DataFrame({
    "Variable": X_test.columns,
    "Mean |SHAP|": np.abs(shap_values).mean(axis=0)
})

importance = importance.sort_values(
    "Mean |SHAP|",
    ascending=False
)

display(importance)

**Interprétation :**

  L'analyse SHAP confirme que le DFG (ckd_epi) est le principal facteur guidant les prédictions du modèle, avec une contribution nettement supérieure à celle des autres variables. L'âge au moment de la mesure (creat_age), certains facteurs génétiques (notamment PKD2 et PKD1 tronquant) ainsi que le sexe féminin apportent également une contribution, mais dans une moindre mesure. Les variables liées aux antécédents d'hypertension et aux événements urologiques ont un impact plus limité sur les prédictions.

## VALIDATION EXTERNE

In [ ]:
# ==========================================================
# PRÉPARATION DU JEU DE VALIDATION EXTERNE
# ==========================================================

# Modèle 1 : Baseline
_, X_external_model1 = prepare_features(
    train_df,
    external_test_df,
    features_model1,
)

# Modèle 2 : Events before 35
_, X_external_model2 = prepare_features(
    train_df,
    external_test_df,
    features_model2,
)

# Modèle 3 : Combiné
_, X_external_model3 = prepare_features(
    train_df,
    external_test_df,
    features_model3,
)

In [ ]:
# ==========================================================
# Construction de la cible de survie
# Jeu de validation externe
# ==========================================================

y_external_test = Surv.from_dataframe(
    event="event_irt",
    time="time_to_event",
    data=external_test_df,
)

In [ ]:
# ==========================================================
# DICTIONNAIRE DES MODÈLES
# ==========================================================

external_models_dict = {

    "Model 1 - baseline": (
        X_train_model1,
        X_external_model1,
    ),

    "Model 2 - before35": (
        X_train_model2,
        X_external_model2,
    ),

    "Model 3 - combined": (
        X_train_model3,
        X_external_model3,
    ),
}

### Random Survival Forest

In [ ]:
# ==========================================================
# RESULTS — EXTERNAL TEST SET
# ==========================================================

external_results_rsf = []

# ==========================================================
# PREDICTIONS
# ==========================================================

external_pred_rsf = []

# ==========================================================
# LOOP OVER THE MODELS
# ==========================================================

for model_name, (X_train, X_external_test) in external_models_dict.items():

    print("\n")
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    # ======================================================
    # FINAL MODEL
    # ======================================================

    rsf = final_rsf_models[model_name]

    # ======================================================
    # SCORES DE RISQUE - TRAIN
    # ======================================================

    train_risk_scores = rsf.predict(X_train)

    # ======================================================
    # HARRELL C-INDEX - TRAIN
    # ======================================================

    train_harrell_cindex = concordance_index_censored(
        event_indicator=y_train["event_irt"],
        event_time=y_train["time_to_event"],
        estimate=train_risk_scores
    )[0]

    # ======================================================
    # SCORES DE RISQUE - TEST EXTERNE
    # ======================================================

    risk_scores = rsf.predict(X_external_test)

    # ======================================================
    # HARRELL C-INDEX
    # ======================================================

    harrell_cindex = concordance_index_censored(
        event_indicator=y_external_test["event_irt"],
        event_time=y_external_test["time_to_event"],
        estimate=risk_scores
    )[0]

    # ======================================================
    # UNO C-INDEX
    # ======================================================

    uno_cindex = concordance_index_ipcw(
        y_train,
        y_external_test,
        risk_scores
    )[0]

    # ======================================================
    # TEMPS MÉDIAN DE SURVIE PRÉDIT
    # ======================================================

    pred_median_survival = predict_median_survival(
        rsf,
        X_external_test
    )

    # ======================================================
    # TABLEAU DES PRÉDICTIONS
    # ======================================================

    df_pred_rsf = pd.DataFrame({

        "patient_id": external_test_df["patient_id"].values,
        "clinical_model": model_name,
        "time_observed": external_test_df["time_to_event"].values,
        "event_irt": external_test_df["event_irt"].values,
        "pred_median_survival": pred_median_survival,

    })

    # ======================================================
    # PATIENTS AYANT PRÉSENTÉ L'ÉVÉNEMENT
    # ET MÉDIANE ESTIMABLE
    # ======================================================

    df_pred_rsf = (
        df_pred_rsf
        .query("event_irt == 1")
        .dropna(subset=["pred_median_survival"])
        .copy()
    )

    # ======================================================
    # ERREURS DE PRÉDICTION
    # ======================================================

    df_pred_rsf["error_rsf"] = (
        df_pred_rsf["pred_median_survival"]
        - df_pred_rsf["time_observed"]
    )

    df_pred_rsf["abs_error_rsf"] = (
        df_pred_rsf["error_rsf"].abs()
    )

    # ======================================================
    # SAUVEGARDE DES PRÉDICTIONS
    # ======================================================

    external_pred_rsf.append(df_pred_rsf)

    # ======================================================
    # MÉTRIQUES
    # ======================================================

    mae = mean_absolute_error(
        df_pred_rsf["time_observed"],
        df_pred_rsf["pred_median_survival"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            df_pred_rsf["time_observed"],
            df_pred_rsf["pred_median_survival"]
        )
    )

    # ======================================================
    # AFFICHAGE
    # ======================================================

    print(f"Train Harrell C-index : {train_harrell_cindex:.4f}")
    print(f"Harrell C-index       : {harrell_cindex:.4f}")
    print(f"Uno C-index           : {uno_cindex:.4f}")
    print(f"Patients évalués      : {len(df_pred_rsf)}")
    print(f"MAE                   : {mae:.2f} années")
    print(f"RMSE                  : {rmse:.2f} années")

    # ======================================================
    # SAUVEGARDE DES RÉSULTATS
    # ======================================================

    external_results_rsf.append({

        "Model": model_name,
        "Train Harrell C-index": round(train_harrell_cindex, 4),
        "Harrell C-index": round(harrell_cindex, 4),
        "Uno C-index": round(uno_cindex, 4),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2)

    })

# ==========================================================
# TABLEAU RÉCAPITULATIF
# ==========================================================

external_results_rsf_df = (
    pd.DataFrame(external_results_rsf)
    .round(4)
)

print("\n")
print("=" * 60)
print("RSF RESULTS — EXTERNAL TEST SET")
print("=" * 60)

display(external_results_rsf_df)

# ==========================================================
# CONCATÉNATION DES PRÉDICTIONS
# ==========================================================

external_pred_rsf_df = pd.concat(
    external_pred_rsf,
    ignore_index=True
)

print("\n")
print("=" * 60)
print("RSF EXTERNAL PREDICTIONS")
print("=" * 60)

display(external_pred_rsf_df.head())

print(f"\nShape : {external_pred_rsf_df.shape}")

## XGBOOST AFT

In [ ]:
# ==========================================================
# JEUX DE DONNÉES ASSOCIÉS AUX MODÈLES CLINIQUES
# VALIDATION EXTERNE
# ==========================================================

EXTERNAL_MODELS = {

    "Baseline": {
        "X_train": X_train_model1,
        "X_test": X_external_model1,
    },

    "Before_35": {
        "X_train": X_train_model2,
        "X_test": X_external_model2,
    },

    "Combined": {
        "X_train": X_train_model3,
        "X_test": X_external_model3,
    }

}

In [ ]:
# ==========================================================
# PRÉDICTIONS DES MODÈLES SUR LE JEU DE VALIDATION EXTERNE
# ==========================================================

# Dictionnaire contenant les prédictions de chaque modèle
# clinique sur le jeu de validation externe.

external_predictions = {}

for model_name, data in EXTERNAL_MODELS.items():

    print("\n" + "=" * 60)
    print(f"PRÉDICTIONS - {model_name}")
    print("=" * 60)

    X_external_test = data["X_test"]

    # ======================================================
    # CRÉATION DE LA DMATRIX DE VALIDATION
    # ======================================================

    dtest = xgb.DMatrix(X_external_test)

    # ======================================================
    # PRÉDICTION DU TEMPS JUSQU'À L'IRT
    # ======================================================

    predicted_time_to_event = final_xgb_models[model_name].predict(dtest)

    # ======================================================
    # RASSEMBLEMENT DES PRÉDICTIONS
    # ======================================================

    predictions_df = pd.DataFrame({

        "event_irt": y_external_test["event_irt"],

        "true_time_to_event": y_external_test["time_to_event"],

        "predicted_time_to_event": predicted_time_to_event

    })

    # Sauvegarde des prédictions.

    external_predictions[model_name] = predictions_df

    # Affichage d'un aperçu des prédictions.

    print(predictions_df.head())



In [ ]:
# ==========================================================
# ÉVALUATION DES MODÈLES XGBOOST SURVIVAL AFT
# SUR LE JEU DE VALIDATION EXTERNE
# ==========================================================

external_results = []

for model_name, data in EXTERNAL_MODELS.items():

    print("\n")
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    # ======================================================
    # DONNÉES D'ENTRAÎNEMENT
    # ======================================================

    X_train = data["X_train"]

    dtrain = xgb.DMatrix(X_train)

    train_predicted_time_to_event = (
        final_xgb_models[model_name]
        .predict(dtrain)
    )

    # ======================================================
    # C-INDEX DE HARRELL - TRAIN
    # ======================================================

    train_harrell_cindex = concordance_index_censored(

        event_indicator=y_train["event_irt"],

        event_time=y_train["time_to_event"],

        estimate=-train_predicted_time_to_event

    )[0]

    # ======================================================
    # RÉCUPÉRATION DES PRÉDICTIONS EXTERNES
    # ======================================================

    predictions_df = external_predictions[model_name]

    predicted_time_to_event = predictions_df[
        "predicted_time_to_event"
    ].to_numpy()

    # ======================================================
    # CALCUL DU C-INDEX DE HARRELL
    # ======================================================

    harrell_cindex = concordance_index_censored(

        event_indicator=y_external_test["event_irt"],

        event_time=y_external_test["time_to_event"],

        estimate=-predicted_time_to_event

    )[0]

    # ======================================================
    # CALCUL DU C-INDEX D'UNO
    # ======================================================

    uno_cindex = concordance_index_ipcw(

        survival_train=y_train,

        survival_test=y_external_test,

        estimate=-predicted_time_to_event

    )[0]

    # ======================================================
    # CALCUL DES ERREURS DE PRÉDICTION
    # (PATIENTS AYANT PRÉSENTÉ L'ÉVÉNEMENT)
    # ======================================================

    event_mask = y_external_test["event_irt"]

    true_time_events = y_external_test["time_to_event"][
        event_mask
    ]

    predicted_time_events = predicted_time_to_event[
        event_mask
    ]

    mae = mean_absolute_error(

        true_time_events,

        predicted_time_events

    )

    rmse = np.sqrt(

        mean_squared_error(

            true_time_events,

            predicted_time_events

        )

    )

    # ======================================================
    # AFFICHAGE
    # ======================================================

    print(f"Train Harrell C-index : {train_harrell_cindex:.4f}")
    print(f"Harrell C-index       : {harrell_cindex:.4f}")
    print(f"Uno C-index           : {uno_cindex:.4f}")
    print(f"Patients évalués      : {event_mask.sum()}")
    print(f"MAE                   : {mae:.2f} années")
    print(f"RMSE                  : {rmse:.2f} années")

    # ======================================================
    # ENREGISTREMENT DES PERFORMANCES
    # ======================================================

    external_results.append({

        "Model": model_name,

        "Train Harrell C-index": round(
            train_harrell_cindex, 4
        ),

        "Harrell C-index": round(
            harrell_cindex, 4
        ),

        "Uno C-index": round(
            uno_cindex, 4
        ),

        "MAE (years)": round(
            mae, 2
        ),

        "RMSE (years)": round(
            rmse, 2
        )

    })

# ==========================================================
# TABLEAU RÉCAPITULATIF DES PERFORMANCES
# ==========================================================

external_results_df = (

    pd.DataFrame(external_results)

    .sort_values(
        "Harrell C-index",
        ascending=False
    )

    .reset_index(drop=True)

    .round(4)

)

print("\n")
print("=" * 80)
print("RÉSULTATS DES MODÈLES XGBOOST SURVIVAL AFT - VALIDATION EXTERNE")
print("=" * 80)

display(external_results_df)

In [ ]:
# ==========================================================
# SAUVEGARDE HTML — TABLEAUX ML (VALIDATION EXTERNE)
# ==========================================================

import os

# ----------------------------------------------------------
# Dossier de sortie
# ----------------------------------------------------------

output_dir = (
    r".\tables\survival_models"
)

os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------------------------
# Chemin du fichier HTML
# ----------------------------------------------------------

html_path = os.path.join(
    output_dir,
    "survival_ml_results_external_ckd_epi.html"
)

# ----------------------------------------------------------
# Génération du HTML
# ----------------------------------------------------------

html_content = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>External Validation - Survival Machine Learning Results</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 40px;
    background-color: white;
}}

h1 {{
    color: #222222;
}}

h2 {{
    margin-top: 50px;
    color: #333333;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 15px;
    margin-bottom: 40px;
}}

th,
td {{
    border: 1px solid #cccccc;
    padding: 8px;
    text-align: center;
}}

th {{
    background-color: #f2f2f2;
}}

tr:nth-child(even) {{
    background-color: #fafafa;
}}

</style>

</head>

<body>

<h1>External Validation of Survival Machine Learning Models</h1>

<h2>Random Survival Forest</h2>

{external_results_rsf_df.to_html(index=False)}

<h2>XGBoost Survival AFT</h2>

{external_results_df.to_html(index=False)}

</body>

</html>
"""

# ----------------------------------------------------------
# Sauvegarde
# ----------------------------------------------------------

with open(
    html_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(html_content)

print(f"HTML sauvegardé :\n{html_path}")